In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from scipy.stats import boxcox
from category_encoders import TargetEncoder


In [2]:
df = pd.read_csv("../data/income_data.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              48842 non-null  int64
 1   workclass        48842 non-null  str  
 2   fnlwgt           48842 non-null  int64
 3   education        48842 non-null  str  
 4   educational-num  48842 non-null  int64
 5   marital-status   48842 non-null  str  
 6   occupation       48842 non-null  str  
 7   relationship     48842 non-null  str  
 8   race             48842 non-null  str  
 9   gender           48842 non-null  str  
 10  capital-gain     48842 non-null  int64
 11  capital-loss     48842 non-null  int64
 12  hours-per-week   48842 non-null  int64
 13  native-country   48842 non-null  str  
 14  income           48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


In [3]:
df.shape

(48842, 15)

In [4]:
df = df.drop_duplicates()

**Feature Engineering**

In [5]:
df['native-country'] = np.where(df['native-country'] == 'United-States', 'United-States', 'Other')
df['capital-neto'] = df['capital-gain']-df['capital-loss']

**Eliminación de Variables**

In [6]:
df = df.drop(columns=['fnlwgt','education','capital-gain','capital-loss'])

**Boxcox y Winsorización**

In [7]:
# 1. Aplicamos la transformación Box-Cox a la variable age
# boxcox nos devuelve la serie transformada y el valor de lambda óptimo encontrado
df['age'], lam = boxcox(df['age'])

In [8]:
# 1. Calculamos los percentiles 1 y 99 para hours-per-week
lower_hours = df['hours-per-week'].quantile(0.01)
upper_hours = df['hours-per-week'].quantile(0.99)

print(f"Límite inferior para el recorte (P1): {lower_hours} horas")
print(f"Límite superior para el recorte (P99): {upper_hours} horas\n")

# 2. Aplicamos la winsorización simétrica usando .clip()
# Guardamos el resultado en una nueva columna para mantener la original intacta
df['hours-per-week'] = df['hours-per-week'].clip(lower=lower_hours, upper=upper_hours)

Límite inferior para el recorte (P1): 8.0 horas
Límite superior para el recorte (P99): 80.0 horas



**División en Train y Test**

In [9]:
X = df.drop(columns=['income'])
y = df['income'].map({'<=50K': 0, '>50K': 1})

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verificamos las dimensiones de los nuevos conjuntos
print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test:  {X_test.shape}")
print("\nDistribución del target en el conjunto de entrenamiento:")
print(y_train.value_counts(normalize=True))

Dimensiones de X_train: (39032, 11)
Dimensiones de X_test:  (9758, 11)

Distribución del target en el conjunto de entrenamiento:
income
0    0.760581
1    0.239419
Name: proportion, dtype: float64


**Encoding**

In [11]:
categoric_cols = X_train.select_dtypes(include ='object').columns.tolist()
categoric_cols

C:\Users\Administrador\AppData\Local\Temp\ipykernel_37760\513441716.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoric_cols = X_train.select_dtypes(include ='object').columns.tolist()


['workclass',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'gender',
 'native-country']

**One-Hot-Encoding**: Se aplicó One-Hot Encoding en marital-status, relationship, native-country, gender y race porque son variables nominales con baja cardinalidad y sin un orden lógico interno. Al transformarlas en columnas binarias independientes (de ceros y unos), el modelo puede interpretar su impacto de forma limpia y directa sin el riesgo de asumir jerarquías artificiales entre categorías como el género o el estado civil.

In [12]:
baja_cardinalidad = ['marital-status', 'relationship', 'native-country', 'gender', 'race']

X_train = pd.get_dummies(X_train, columns=baja_cardinalidad, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=baja_cardinalidad, drop_first=True, dtype=int)

**Target Encoding**: Se aplicó Target Encoding en occupation y workclass debido a su alta cardinalidad (15 y 7 categorías respectivamente), evitando así multiplicar innecesariamente las columnas del dataset si hubiéramos usado One-Hot Encoding. En su lugar, ambas variables se reducen a una única columna numérica que reemplaza cada texto por su probabilidad real de ganar más de 50K, simplificando la estructura de los datos y maximizando la señal predictiva del modelo.

In [13]:
# 2. MÉTODO 1: Target Encoding (Para variables con muchas categorías / Alta Cardinalidad)
# Reemplaza 'occupation' y 'workclass' por la media del target en Train
alta_cardinalidad = ['occupation', 'workclass']

encoder_target = TargetEncoder(cols=alta_cardinalidad)
# El fit se hace SOLO en Train, y se aplica (transform) en ambos
X_train = encoder_target.fit_transform(X_train, y_train)
X_test = encoder_target.transform(X_test)

Comprobación de que tienen las mismas columnas

In [14]:
# 4. Alineación final (Garantiza que Train y Test tengan las mismas columnas)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"Dimensiones finales de X_train: {X_train.shape}")
print("Columnas listas para el modelo:")
print(X_train.columns.tolist())

Dimensiones finales de X_train: (39032, 23)
Columnas listas para el modelo:
['age', 'workclass', 'educational-num', 'occupation', 'hours-per-week', 'capital-neto', 'marital-status_Married-AF-spouse', 'marital-status_Married-civ-spouse', 'marital-status_Married-spouse-absent', 'marital-status_Never-married', 'marital-status_Separated', 'marital-status_Widowed', 'relationship_Not-in-family', 'relationship_Other-relative', 'relationship_Own-child', 'relationship_Unmarried', 'relationship_Wife', 'native-country_United-States', 'gender_Male', 'race_Asian-Pac-Islander', 'race_Black', 'race_Other', 'race_White']


In [15]:
# 1. Concatenamos X e y para Train y Test respectivamente
df_train_final = pd.concat([X_train, y_train], axis=1)
df_test_final = pd.concat([X_test, y_test], axis=1)

# 2. Guardamos en la carpeta de datos (asegúrate de que la ruta exista)
# Usamos index=False para que no te cree una columna extra e incómoda con los índices antiguos
df_train_final.to_csv("../data/train.csv", index=False)
df_test_final.to_csv("../data/test.csv", index=False)

print("¡Archivos guardados con éxito en la carpeta '../data/'!")
print(f"Registros en Train: {df_train_final.shape[0]} | Columnas: {df_train_final.shape[1]}")
print(f"Registros en Test:  {df_test_final.shape[0]} | Columnas: {df_test_final.shape[1]}")

¡Archivos guardados con éxito en la carpeta '../data/'!
Registros en Train: 39032 | Columnas: 24
Registros en Test:  9758 | Columnas: 24
